# Iris Flower Morphological Measurements by Species Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [Iris Flower Morphological Measurements by Species](https://sen.science/doi/10.71728/senscience.62ya-kx66) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset URL
url = 'https://sen.science/doi/10.71728/senscience.62ya-kx66/fair2.json'

# Load the dataset metadata from Croissant
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")
print(f"Keywords: {getattr(metadata, 'keywords', [])}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
print("--- Record Sets Overview ---\n")
# Find all record set @ids
record_sets = [r['@id'] for r in getattr(metadata, 'recordSet', [])]
if record_sets:
    for rid in record_sets:
        print(f'RecordSet @id: {rid}')
else:
    print("No explicit recordSet listed in metadata. Attempting to enumerate from schema...")
    # Try to find record sets declared in the underlying schema
    # This method works if the underlying mlcroissant has parsed from JSON-LD
    schema_dict = getattr(metadata, 'to_json', lambda: {{}})()
    if 'recordSet' in schema_dict:
        if isinstance(schema_dict['recordSet'], list):
            record_sets = [r['@id'] for r in schema_dict['recordSet']]
        elif isinstance(schema_dict['recordSet'], dict):
            record_sets = [schema_dict['recordSet']['@id']]
        for rid in record_sets:
            print(f'RecordSet @id: {rid}')
    else:
        print("Unable to identify record sets directly. For this dataset, typically it contains only one, inferred from the data file distribution.")
        record_sets = ['cr:recordSet/iris']  # fallback to conventional Croissant id if known

print("\n--- Fields for each Record Set ---\n")
for rid in record_sets:
    print(f'RecordSet {rid}: Fields:')
    for f in dataset.fields(record_set=rid):
        print(f"  Field @id: {f['@id']}, name: {f.get('name', 'N/A')}, type: {f.get('dataType', 'N/A')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# We'll use the first listed record set, or fallback to 'cr:recordSet/iris' which is commonly used for the IRIS dataset Croissant schema.
record_sets = record_sets if record_sets else ['cr:recordSet/iris']
dataframes = {}

for record_set in record_sets:
    print(f'Extracting records for RecordSet @id: {record_set}')
    records = list(dataset.records(record_set=record_set))
    if len(records) == 0:
        print(f"No records found for {record_set}. Skipping.")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set] = df
    print(f"Columns found: {df.columns.tolist()}")
    display(df.head())

# For further processing, pick the main data record set. For Iris Croissant, it's often 'cr:recordSet/iris'.
main_record_set = record_sets[0]
print(f"Chosen main record set for EDA: {main_record_set}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, and grouping data.

In [ ]:
# Let's pick a numeric field, e.g., 'sepal_length' or as referenced in the schema by its @id.

df = dataframes[main_record_set]
numeric_fields = [col for col in df.columns if 'length' in col or 'width' in col]
print(f"Numeric measurement fields: {numeric_fields}")

# We'll use 'sepal_length' or a similar variant—the exact column name as loaded
for col in ['sepal_length', 'sepalLength', 'sepal length (cm)']:
    if col in df.columns:
        numeric_field = col
        break
else:
    # Take the first measurement column
    numeric_field = numeric_fields[0]

# Filter for sepal_length > threshold (e.g., 5.0 cm)
threshold = 5.0
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold} (showing up to 5):")
display(filtered_df.head())

# Normalize the numeric field (z-score)
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by species field
# Typical Iris field: 'species' or 'class' or similar (check exact column via printing columns)
for gfield in ['species', 'Species', 'class', 'target']:
    if gfield in df.columns:
        group_field = gfield
        break
else:
    group_field = df.columns[-1]  # Fallback to last column

if group_field in df.columns:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"\nGrouped data by {group_field} (showing means):")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of sepal length by species
plt.figure(figsize=(8,5))
sns.histplot(df, x=numeric_field, hue=group_field, kde=True, bins=15, palette="Set2")
plt.title(f"Distribution of {numeric_field} by {group_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Pairplot for all measurement fields colored by species
sns.pairplot(df, vars=numeric_fields, hue=group_field, diag_kind='kde', height=2, palette="Set1")
plt.suptitle("Pairwise distributions by Species", y=1.02)
plt.show()

## 6. Conclusion
This notebook demonstrates how to use `mlcroissant` to load and inspect the [Iris dataset](https://sen.science/doi/10.71728/senscience.62ya-kx66). After exploring metadata, fields, and records, we filtered and normalized sepal length values, grouped by species, and visualized feature distributions. This process is easily extensible to other Croissant-based datasets! Explore additional fields and try other visualizations for deeper insights.